# Part 2: Web Search — Inverted Index & Search Engine
Implements an inverted index over a set of webpages and
answers TF-IDF-based search queries.

In [ ]:
import os, re, math, string
os.makedirs("webpages", exist_ok=True)

sample_pages = {
    "page1.txt": "Data structures is the study of structures for storing data.",
    "page2.txt": "Structural engineers collect data about structures.",
    "page3.txt": "Algorithms and data structures are fundamental to computer science.",
    "page4.txt": "Search engines use inverted index structures for fast data retrieval.",
    "page5.txt": "Machine learning algorithms process large amounts of data efficiently."
}

for name, content in sample_pages.items():
    with open(f"webpages/{name}", "w") as f:
        f.write(content)

# ──────────────────────────────────────────────────────────────────────
# Create sample actions.txt
# ──────────────────────────────────────────────────────────────────────
actions_content = """addPage page1.txt
addPage page2.txt
addPage page3.txt
addPage page4.txt
addPage page5.txt
queryFindPagesWhichContainWord data
queryFindPagesWhichContainWord structure
queryFindPagesWhichContainWord algorithm
queryFindPagesWhichContainWord quantum
queryFindPositionsOfWordInAPage data page1.txt
queryFindPositionsOfWordInAPage structure page2.txt
queryFindPositionsOfWordInAPage data page9.txt
"""

with open("actions.txt", "w") as f:
    f.write(actions_content)

print("Sample webpages and actions.txt created successfully.")
print("Pages:", list(sample_pages.keys()))

Sample webpages and actions.txt created successfully.
Pages: ['page1.txt', 'page2.txt', 'page3.txt', 'page4.txt', 'page5.txt']


In [ ]:
STOP_WORDS = {
    'a', 'an', 'the', 'they', 'these', 'this', 'for',
    'is', 'are', 'was', 'of', 'or', 'and', 'does', 'will', 'whose'
}
PUNCTUATION = set("{}[]<>=(). ,;'\"?#!-:")
PLURAL_MAP = {
    'stacks'        : 'stack',
    'structures'    : 'structure',
    'applications'  : 'application',
    'algorithms'    : 'algorithm',
    'engineers'     : 'engineer',
    'engines'       : 'engine',
    'amounts'       : 'amount',
    'systems'       : 'system',
}

def normalise(word):
    """Lowercase, strip punctuation, and resolve plural forms."""
    word = word.lower()
    # Replace punctuation chars with empty string
    word = ''.join('' if ch in PUNCTUATION else ch for ch in word)
    word = word.strip()
    # Map plural to singular
    return PLURAL_MAP.get(word, word)


print("normalise('Structures') →", normalise('Structures'))
print("normalise('data.')     →", normalise('data.'))
print("normalise('Engineers') →", normalise('Engineers'))

normalise('Structures') → structure
normalise('data.')     → data
normalise('Engineers') → engineer


In [ ]:
class MySet:

    def __init__(self):
        self._data = set()

    def addElement(self, element):
        """Add element to the set."""
        self._data.add(element)

    def union(self, otherSet):
        """Return a new MySet representing the union."""
        result = MySet()
        result._data = self._data | otherSet._data
        return result

    def intersection(self, otherSet):
        """Return a new MySet representing the intersection."""
        result = MySet()
        result._data = self._data & otherSet._data
        return result

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"MySet({self._data})"


# Quick test
s1 = MySet(); s1.addElement('a'); s1.addElement('b')
s2 = MySet(); s2.addElement('b'); s2.addElement('c')
print("Union       :", s1.union(s2))
print("Intersection:", s1.intersection(s2))

Union       : MySet({'c', 'b', 'a'})
Intersection: MySet({'b'})


In [ ]:
class Position:
    def __init__(self, page_entry, word_index: int):

        self._page_entry  = page_entry
        self._word_index  = word_index

    def getPageEntry(self):
        """Return the associated PageEntry."""
        return self._page_entry

    def getWordIndex(self):
        """Return the 1-based word index within the page."""
        return self._word_index

    def __repr__(self):
        return f"Position({self._page_entry.pageName}, idx={self._word_index})"

    def __hash__(self):
        return hash((self._page_entry.pageName, self._word_index))

    def __eq__(self, other):
        return (self._page_entry.pageName == other._page_entry.pageName and
                self._word_index == other._word_index)

print("Position class defined.")

Position class defined.


In [ ]:
class WordEntry:

    def __init__(self, word: str):
        """
        Args:
            word : str — the word this entry tracks
        """
        self.word       = word
        self._positions = []   # list of Position objects

    def addPosition(self, position: Position):
        """Add a single position entry."""
        self._positions.append(position)

    def addPositions(self, positions: list):
        """Add multiple position entries."""
        self._positions.extend(positions)

    def getAllPositionsForThisWord(self):
        """Return a list of all Position objects for this word."""
        return self._positions

    def getTermFrequency(self, page_entry):

        count = sum(1 for pos in self._positions
                    if pos.getPageEntry().pageName == page_entry.pageName)
        total_words = page_entry.totalWords
        if total_words == 0:
            return 0.0
        return count / total_words

    def __repr__(self):
        return f"WordEntry('{self.word}', positions={len(self._positions)})"

print("WordEntry class defined.")

WordEntry class defined.


In [ ]:
class PageIndex:


    def __init__(self):
        # word (str) → WordEntry
        self._index = {}

    def addPositionForWord(self, word: str, position: Position):

        if word not in self._index:
            self._index[word] = WordEntry(word)
        self._index[word].addPosition(position)

    def getWordEntries(self):
        """Return a list of all WordEntry objects in this page index."""
        return list(self._index.values())

    def getWordEntry(self, word: str):
        """Return the WordEntry for a specific word, or None."""
        return self._index.get(word, None)

    def __repr__(self):
        return f"PageIndex(words={list(self._index.keys())})"

print("PageIndex class defined.")

PageIndex class defined.


In [ ]:
class PageEntry:

    WEBPAGES_DIR = "webpages"   # folder where page files live

    def __init__(self, pageName: str):

        self.pageName   = pageName
        self.totalWords = 0        # total number of words (including stop words)
        self._pageIndex = PageIndex()
        self._build_index()

    def _build_index(self):

        filepath = os.path.join(self.WEBPAGES_DIR, self.pageName)
        with open(filepath, 'r') as f:
            text = f.read()

        # Replace punctuation with spaces
        for ch in PUNCTUATION:
            text = text.replace(ch, ' ')

        # Split into raw tokens
        raw_tokens = text.split()
        self.totalWords = len(raw_tokens)  # count all tokens for TF denominator

        # Build index; position index is 1-based and counts ALL words
        for idx, token in enumerate(raw_tokens, start=1):
            word = normalise(token)
            if not word:             # empty after stripping punctuation
                continue
            if word in STOP_WORDS:   # skip stop words from index
                continue
            position = Position(self, idx)
            self._pageIndex.addPositionForWord(word, position)

    def getPageIndex(self):
        """Return the PageIndex for this page."""
        return self._pageIndex

    def __repr__(self):
        return f"PageEntry('{self.pageName}')"


# Quick test
pe = PageEntry('page1.txt')
print("Page:", pe)
print("Total words:", pe.totalWords)
print("Indexed words:", [we.word for we in pe.getPageIndex().getWordEntries()])

Page: PageEntry('page1.txt')
Total words: 10
Indexed words: ['data', 'structure', 'study', 'storing']


In [ ]:
class MyHashTable:


    def __init__(self, capacity: int = 1024):

        self._capacity = capacity
        self._table    = [[] for _ in range(capacity)]   # list of chains

    def getHashIndex(self, word: str) -> int:

        return hash(word) % self._capacity

    def _get_entry(self, word: str):
        """Internal: return the existing WordEntry for `word`, or None."""
        bucket = self._table[self.getHashIndex(word)]
        for entry in bucket:
            if entry.word == word:
                return entry
        return None

    def addPositionsForWord(self, w: 'WordEntry'):

        existing = self._get_entry(w.word)
        if existing is None:
            # Insert new entry
            bucket = self._table[self.getHashIndex(w.word)]
            bucket.append(w)
        else:
            # Merge positions into the existing entry
            existing.addPositions(w.getAllPositionsForThisWord())

    def getWordEntry(self, word: str):
        """Retrieve the WordEntry for `word`, or None if not present."""
        return self._get_entry(word)

    def getAllWordEntries(self):
        """Return a flat list of all WordEntry objects in the table."""
        result = []
        for bucket in self._table:
            result.extend(bucket)
        return result


# Quick test
ht = MyHashTable()
we = WordEntry('data')
pe_test = PageEntry('page1.txt')
we.addPosition(Position(pe_test, 1))
ht.addPositionsForWord(we)
print("Retrieved entry:", ht.getWordEntry('data'))

Retrieved entry: WordEntry('data', positions=1)


In [ ]:
class InvertedPageIndex:

    def __init__(self):
        self._hashtable = MyHashTable(capacity=2048)
        self._pages     = {}   # pageName → PageEntry

    def addPage(self, page_entry: PageEntry):

        self._pages[page_entry.pageName] = page_entry
        for word_entry in page_entry.getPageIndex().getWordEntries():
            self._hashtable.addPositionsForWord(word_entry)

    def getPagesWhichContainWord(self, word: str) -> MySet:

        word = normalise(word)
        result = MySet()
        entry = self._hashtable.getWordEntry(word)
        if entry is not None:
            for pos in entry.getAllPositionsForThisWord():
                result.addElement(pos.getPageEntry())
        return result

    def getPositionsOfWordInPage(self, word: str, pageName: str):

        if pageName not in self._pages:
            return None   # page not in database
        word = normalise(word)
        entry = self._hashtable.getWordEntry(word)
        if entry is None:
            return []
        indices = sorted(
            pos.getWordIndex()
            for pos in entry.getAllPositionsForThisWord()
            if pos.getPageEntry().pageName == pageName
        )
        return indices

    def getAllPages(self):
        """Return dict of all indexed pages."""
        return self._pages

print("InvertedPageIndex class defined.")

InvertedPageIndex class defined.


In [ ]:
def tfidf_score(word: str, page_entry: PageEntry, inverted_index: InvertedPageIndex) -> float:

    word_norm = normalise(word)
    N  = len(inverted_index.getAllPages())          # total number of pages
    pages_with_word = inverted_index.getPagesWhichContainWord(word_norm)
    n_w = len(pages_with_word)

    if n_w == 0:
        return 0.0

    # TF: occurrences of word in this page / total words in this page
    entry = inverted_index._hashtable.getWordEntry(word_norm)
    count_in_page = sum(
        1 for pos in entry.getAllPositionsForThisWord()
        if pos.getPageEntry().pageName == page_entry.pageName
    ) if entry else 0

    tf  = count_in_page / page_entry.totalWords if page_entry.totalWords > 0 else 0.0
    idf = math.log(N / n_w)
    return tf * idf


print("tfidf_score function defined.")

tfidf_score function defined.


In [ ]:
class SearchEngine:
    """
    Main search engine class.
    Wraps an InvertedPageIndex and processes action commands.
    """

    def __init__(self):
        """Create an empty InvertedPageIndex."""
        self._index = InvertedPageIndex()

    def performAction(self, actionMessage: str):

        parts = actionMessage.strip().split()
        if not parts:
            return

        action = parts[0]

        # ── addPage ──────────────────────────────────────────────────
        if action == 'addPage':
            pageName = parts[1]
            page_entry = PageEntry(pageName)
            self._index.addPage(page_entry)
            print(f"Added page: {pageName}")

        # ── queryFindPagesWhichContainWord ───────────────────────────
        elif action == 'queryFindPagesWhichContainWord':
            word = parts[1]
            pages = self._index.getPagesWhichContainWord(word)
            if len(pages) == 0:
                print(f"No webpage contains word {word}")
            else:
                names = sorted(pe.pageName for pe in pages)
                print(", ".join(names))

        # ── queryFindPositionsOfWordInAPage ──────────────────────────
        elif action == 'queryFindPositionsOfWordInAPage':
            word     = parts[1]
            pageName = parts[2]
            positions = self._index.getPositionsOfWordInPage(word, pageName)
            if positions is None:
                print(f"No webpage {pageName} found")
            elif len(positions) == 0:
                print(f"Webpage {pageName} does not contain word {word}")
            else:
                print(", ".join(map(str, positions)))

        else:
            print(f"Unknown action: {action}")

print("SearchEngine class defined.")

SearchEngine class defined.


In [ ]:
print("=" * 60)
print("Processing actions.txt")
print("=" * 60)

engine = SearchEngine()

with open('actions.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            print(f"\n> {line}")
            engine.performAction(line)

Processing actions.txt

> addPage page1.txt
Added page: page1.txt

> addPage page2.txt
Added page: page2.txt

> addPage page3.txt
Added page: page3.txt

> addPage page4.txt
Added page: page4.txt

> addPage page5.txt
Added page: page5.txt

> queryFindPagesWhichContainWord data
page1.txt, page2.txt, page3.txt, page4.txt, page5.txt

> queryFindPagesWhichContainWord structure
page1.txt, page2.txt, page3.txt, page4.txt

> queryFindPagesWhichContainWord algorithm
page3.txt, page5.txt

> queryFindPagesWhichContainWord quantum
No webpage contains word quantum

> queryFindPositionsOfWordInAPage data page1.txt
1, 10

> queryFindPositionsOfWordInAPage structure page2.txt
6

> queryFindPositionsOfWordInAPage data page9.txt
No webpage page9.txt found


In [ ]:
def ranked_search(query: str, engine: SearchEngine, top_n: int = 5):
    """
    Given a multi-word query, rank pages by their total TF-IDF score
    (sum of per-word TF-IDF for all query words that appear in the page).
    """
    query_words = [normalise(w) for w in query.split()
                   if normalise(w) and normalise(w) not in STOP_WORDS]

    # Score each page
    scores = {}
    for word in query_words:
        pages = engine._index.getPagesWhichContainWord(word)
        for page_entry in pages:
            s = tfidf_score(word, page_entry, engine._index)
            scores[page_entry.pageName] = scores.get(page_entry.pageName, 0.0) + s

    if not scores:
        print(f"No results for query: '{query}'")
        return

    # Sort by score descending
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print(f"\nRanked results for: '{query}'")
    print("-" * 40)
    for rank, (page, score) in enumerate(ranked[:top_n], 1):
        print(f"  {rank}. {page:20s}  score={score:.6f}")


print("=" * 60)
print("TF-IDF Ranked Search Demo")
print("=" * 60)
ranked_search("data structures", engine)
ranked_search("algorithm", engine)
ranked_search("search engine", engine)

TF-IDF Ranked Search Demo

Ranked results for: 'data structures'
----------------------------------------
  1. page1.txt             score=0.044629
  2. page2.txt             score=0.037191
  3. page3.txt             score=0.024794
  4. page4.txt             score=0.022314
  5. page5.txt             score=0.000000

Ranked results for: 'algorithm'
----------------------------------------
  1. page3.txt             score=0.101810
  2. page5.txt             score=0.101810

Ranked results for: 'search engine'
----------------------------------------
  1. page4.txt             score=0.321888
